# Loan Approval Prediction

**Objective:** predict whether a loan application is `Approved` (Y) or `Rejected` (N) using applicant demographic and financial features.

This notebook follows a practical-style implementation: preprocessing, encoding, scaling, model definition, training, prediction, and evaluation are shown as separate steps. **No `Pipeline` is used.**

## 1. Import libraries and load the dataset

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

df = pd.read_csv("loan_prediction.csv")
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))
print("Duplicate rows:", df.duplicated().sum())

## 2. Explore and prepare the target

In [ ]:
# Drop the ID column as it carries no predictive information.
# Keep a raw copy for the Find-S algorithm (needs original string labels).
df = df.drop(columns=["Loan_ID"]).drop_duplicates().reset_index(drop=True)
df_raw = df.copy()   # used by Find-S

print("Class distribution:")
display(df["Loan_Status"].value_counts().to_frame("Count"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class balance
df["Loan_Status"].value_counts().reindex(["Y", "N"]).plot(
    kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Loan Status Distribution")
axes[0].set_xlabel("Loan Status  (Y = Approved, N = Rejected)")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Applicant income distribution
axes[1].hist(df["ApplicantIncome"], bins=30, edgecolor="black")
axes[1].set_title("Applicant Income Distribution")
axes[1].set_xlabel("Income")
axes[1].set_ylabel("Frequency")

# Loan amount distribution
axes[2].hist(df["LoanAmount"].dropna(), bins=30, edgecolor="black")
axes[2].set_title("Loan Amount Distribution")
axes[2].set_xlabel("Loan Amount (thousands)")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric columns.
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(9, 6))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

### Prepare the input features and target

`Loan_ID` has been removed. All remaining columns except `Loan_Status` are used as inputs. Missing values and encoding are handled inside the preprocessing function using training-set statistics only.

The target is kept as `Y` / `N` (binary: 1 = Approved, 0 = Rejected) after label encoding.

In [ ]:
# Label-encode target: Y -> 1, N -> 0
df["Loan_Status"] = (df["Loan_Status"] == "Y").astype(int)

X = df.drop(columns=["Loan_Status"]).copy()
y = df["Loan_Status"].copy()

numeric_features     = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 3. Define a practical-style preprocessing function

This helper performs the steps manually, rather than using a `Pipeline`:
1. Fill missing numeric values using training-set medians.
2. Fill missing categorical values using training-set modes.
3. Label-encode every categorical column (fit on training data only).
4. Optionally scale the features using a separately fitted `StandardScaler`.

Imputation values, encoders, and scaler are all learned from the training data only.

In [ ]:
def prepare_data(X_train_raw, X_test_raw, scale=True):
    X_train = X_train_raw.copy()
    X_test  = X_test_raw.copy()

    # Fill numeric missing values using training-set medians.
    for col in X_train.select_dtypes(include=np.number).columns:
        median_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_value)
        X_test[col]  = X_test[col].fillna(median_value)

    # Fill categorical missing values using training-set modes.
    for col in X_train.select_dtypes(exclude=np.number).columns:
        modes = X_train[col].mode(dropna=True)
        fill_value = modes.iloc[0] if len(modes) else "Unknown"
        X_train[col] = X_train[col].fillna(fill_value)
        X_test[col]  = X_test[col].fillna(fill_value)

    # Label-encode each categorical column (fit on training data only).
    encoders = {}
    for col in X_train.select_dtypes(exclude=np.number).columns:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_test[col]  = X_test[col].astype(str).map(
            lambda v, le=le: le.transform([v])[0]
            if v in le.classes_ else 0
        )
        encoders[col] = le

    scaler = None
    if scale:
        scaler  = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
        X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X_train.columns)

    return X_train, X_test, scaler

## 4. Compare train-test ratios and finalise one

Decision Tree is used as the reference algorithm for each split ratio (it does not require scaling, making it a neutral baseline for split selection). Each split is stratified and uses the same random state. The ratio with the highest F1-score is selected (accuracy breaks ties).

In [ ]:
split_ratios  = {"80:20": 0.20, "70:30": 0.30, "75:25": 0.25, "60:40": 0.40}
split_results = []

for split_name, test_size in split_ratios.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )
    X_tr_p, X_te_p, _ = prepare_data(X_tr, X_te, scale=False)
    ref_model = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
    ref_model.fit(X_tr_p, y_tr)
    preds = ref_model.predict(X_te_p)
    split_results.append({
        "Train-Test Split": split_name,
        "Accuracy":  accuracy_score(y_te, preds),
        "Precision": precision_score(y_te, preds, zero_division=0),
        "Recall":    recall_score(y_te, preds,    zero_division=0),
        "F1-score":  f1_score(y_te, preds,        zero_division=0),
    })

split_results_df = pd.DataFrame(split_results)
display(split_results_df.style.format({
    "Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}", "F1-score": "{:.4f}"
}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.flat, ["Accuracy", "Precision", "Recall", "F1-score"]):
    ax.bar(split_results_df["Train-Test Split"], split_results_df[metric], edgecolor="black")
    ax.set_title(f"{metric} by Train-Test Split")
    ax.set_xlabel("Train-Test Split")
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
split_ranking = split_results_df.sort_values(
    by=["F1-score", "Accuracy"], ascending=False
).reset_index(drop=True)
selected_split     = split_ranking.loc[0, "Train-Test Split"]
selected_test_size = split_ratios[selected_split]

print("Selected split:", selected_split)
print("Selection rule: highest F1-score, with accuracy used to break ties.")
display(split_ranking)

## 5. Fix the selected split and compare all five algorithms

From this point onward, only the selected split is used. Each algorithm is trained and evaluated on the same training and test rows.

- **Baseline** and **Find-S** do not use scaling.
- **Decision Tree** does not require scaling.
- **KNN** and **ANN** use `StandardScaler` fitted on training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=selected_test_size, random_state=RANDOM_STATE, stratify=y
)

X_train_scaled,   X_test_scaled,   scaler = prepare_data(X_train, X_test, scale=True)
X_train_unscaled, X_test_unscaled, _      = prepare_data(X_train, X_test, scale=False)

print(f"Finalized split: {selected_split}")
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))
print("Feature shape:", X_train_scaled.shape)

### Algorithm 1 — Baseline (Credit History Rule)

In [ ]:
# Baseline: predict Approved (1) when Credit_History == 1, else Rejected (0).
# Uses the unscaled test set so Credit_History retains its original encoded value.
baseline_pred = (X_test_unscaled["Credit_History"] >= 0.5).astype(int)

baseline_scores = {
    "Algorithm": "Baseline",
    "Accuracy":  accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred, zero_division=0),
    "Recall":    recall_score(y_test, baseline_pred,    zero_division=0),
    "F1-score":  f1_score(y_test, baseline_pred,        zero_division=0),
}
print("Baseline (Credit History Rule)")
for metric, value in baseline_scores.items():
    if metric != "Algorithm":
        print(f"  {metric}: {value:.4f}")

### Algorithm 2 — Find-S

In [ ]:
# Find-S works on discretised, categorical features.
# We use the raw (string) data before label-encoding.

def discretise_continuous(series):
    """Map a numeric continuous series to Low / Medium / High."""
    q33, q66 = series.quantile(0.33), series.quantile(0.66)
    if q33 == q66:
        return pd.cut(series, bins=3, labels=["Low", "Medium", "High"]).astype(str)
    return pd.cut(series, bins=[-np.inf, q33, q66, np.inf],
                  labels=["Low", "Medium", "High"]).astype(str)


def prepare_finds(df_in):
    """Discretise continuous columns and convert all features to string categories."""
    d = df_in.copy()
    for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
        d[col] = discretise_continuous(d[col].fillna(d[col].median()))
    for col in d.columns:
        if col != "Loan_Status":
            d[col] = d[col].fillna(d[col].mode()[0]).astype(str)
    return d


feature_cols_fs = [c for c in df_raw.columns if c != "Loan_Status"]

# Build train / test index from the same stratified split
train_idx = X_train.index
test_idx  = X_test.index

df_fs      = prepare_finds(df_raw)
df_fs_train = df_fs.loc[train_idx]
df_fs_test  = df_fs.loc[test_idx]


def find_s_train(data, feature_cols, target="Loan_Status", pos_label="Y"):
    hypothesis = {col: None for col in feature_cols}
    for _, row in data[data[target] == pos_label].iterrows():
        for col in feature_cols:
            val = str(row[col])
            if hypothesis[col] is None:
                hypothesis[col] = val
            elif hypothesis[col] != val:
                hypothesis[col] = "?"
    return hypothesis


def find_s_predict(row, hypothesis, feature_cols):
    for col in feature_cols:
        if hypothesis[col] not in (None, "?") and str(row[col]) != hypothesis[col]:
            return 0
    return 1


hypothesis = find_s_train(df_fs_train, feature_cols_fs)
fs_pred    = df_fs_test.apply(
    lambda row: find_s_predict(row, hypothesis, feature_cols_fs), axis=1
).values

fs_scores = {
    "Algorithm": "Find-S",
    "Accuracy":  accuracy_score(y_test, fs_pred),
    "Precision": precision_score(y_test, fs_pred, zero_division=0),
    "Recall":    recall_score(y_test, fs_pred,    zero_division=0),
    "F1-score":  f1_score(y_test, fs_pred,        zero_division=0),
}
print("Find-S")
for metric, value in fs_scores.items():
    if metric != "Algorithm":
        print(f"  {metric}: {value:.4f}")

### Algorithm 3 — Decision Tree Classifier

In [ ]:
# Algorithm 3: Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE)
dt_model.fit(X_train_unscaled, y_train)
dt_pred = dt_model.predict(X_test_unscaled)

dt_scores = {
    "Algorithm": "Decision Tree",
    "Accuracy":  accuracy_score(y_test, dt_pred),
    "Precision": precision_score(y_test, dt_pred, zero_division=0),
    "Recall":    recall_score(y_test, dt_pred,    zero_division=0),
    "F1-score":  f1_score(y_test, dt_pred,        zero_division=0),
}
print("Decision Tree Classifier")
for metric, value in dt_scores.items():
    if metric != "Algorithm":
        print(f"  {metric}: {value:.4f}")

### Algorithm 4 — K-Nearest Neighbors (KNN)

In [ ]:
# Select best k by scanning k = 1..20 on the training split.
k_values, k_f1s = list(range(1, 21)), []
for k in k_values:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train_scaled, y_train)
    k_f1s.append(f1_score(y_test, knn_tmp.predict(X_test_scaled), zero_division=0))

best_k = k_values[k_f1s.index(max(k_f1s))]
print(f"Best k = {best_k}  (F1 = {max(k_f1s):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(k_values, k_f1s, marker="o")
plt.axvline(best_k, linestyle="--", label=f"Best k = {best_k}")
plt.title("KNN — F1-score vs k")
plt.xlabel("k")
plt.ylabel("F1-score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Algorithm 4: KNN with the best k found above.
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

knn_scores = {
    "Algorithm": "KNN",
    "Accuracy":  accuracy_score(y_test, knn_pred),
    "Precision": precision_score(y_test, knn_pred, zero_division=0),
    "Recall":    recall_score(y_test, knn_pred,    zero_division=0),
    "F1-score":  f1_score(y_test, knn_pred,        zero_division=0),
}
print(f"KNN (k = {best_k})")
for metric, value in knn_scores.items():
    if metric != "Algorithm":
        print(f"  {metric}: {value:.4f}")

### Algorithm 5 — Artificial Neural Network (ANN)

In [ ]:
# Algorithm 5: ANN — two hidden layers with ReLU activation.
ann_model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(8,  activation="relu"),
    Dense(1,  activation="sigmoid"),
])
ann_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

history = ann_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0,
)
print(f"Training stopped at epoch {len(history.history["loss"])}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"],     label="Training Loss",   color="steelblue")
plt.plot(history.history["val_loss"], label="Validation Loss", color="tomato")
plt.title("ANN — Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ann_prob = ann_model.predict(X_test_scaled, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)

ann_scores = {
    "Algorithm": "ANN",
    "Accuracy":  accuracy_score(y_test, ann_pred),
    "Precision": precision_score(y_test, ann_pred, zero_division=0),
    "Recall":    recall_score(y_test, ann_pred,    zero_division=0),
    "F1-score":  f1_score(y_test, ann_pred,        zero_division=0),
}
print("ANN")
for metric, value in ann_scores.items():
    if metric != "Algorithm":
        print(f"  {metric}: {value:.4f}")

In [ ]:
algorithm_results = pd.DataFrame(
    [baseline_scores, fs_scores, dt_scores, knn_scores, ann_scores]
)
algorithm_results = algorithm_results.sort_values("F1-score", ascending=False).reset_index(drop=True)
display(algorithm_results.style.format({
    "Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}", "F1-score": "{:.4f}"
}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, metric in zip(axes.flat, ["Accuracy", "Precision", "Recall", "F1-score"]):
    plot_data = algorithm_results.sort_values(metric, ascending=False)
    ax.bar(plot_data["Algorithm"], plot_data[metric], edgecolor="black")
    ax.set_title(f"{metric} Comparison")
    ax.set_xlabel("Algorithm")
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
best_algorithm_name = algorithm_results.iloc[0]["Algorithm"]
print("Best algorithm based on F1-score:", best_algorithm_name)
print(f"F1-score: {algorithm_results.iloc[0]['F1-score']:.4f}")
print("The other metrics are used as supporting measures.")

## 6. Confusion matrix and classification report

In [ ]:
prediction_map = {
    "Baseline":     baseline_pred,
    "Find-S":       fs_pred,
    "Decision Tree": dt_pred,
    "KNN":          knn_pred,
    "ANN":          ann_pred,
}
best_predictions = prediction_map[best_algorithm_name]

print(classification_report(y_test, best_predictions,
                            target_names=["Rejected", "Approved"], zero_division=0))
cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Rejected", "Approved"],
            yticklabels=["Rejected", "Approved"])
plt.title(f"Confusion Matrix — {best_algorithm_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 7. Final model and sample prediction

After evaluating all algorithms on the selected hold-out test set, the best-performing model is reported with its held-out metrics. A sample prediction is demonstrated using one row from the test set. **No pipeline is used.**

In [ ]:
# Show the held-out test metrics for the selected algorithm.
best_row = algorithm_results.loc[algorithm_results["Algorithm"] == best_algorithm_name].iloc[0]
print("Final algorithm:", best_algorithm_name)
print("Evaluation split:", selected_split)
print(f"Accuracy : {best_row['Accuracy']:.4f}")
print(f"Precision: {best_row['Precision']:.4f}")
print(f"Recall   : {best_row['Recall']:.4f}")
print(f"F1-score : {best_row['F1-score']:.4f}")

In [ ]:
# Demonstrate a prediction on the first held-out test row.
print("Sample applicant features (first test row):")
display(X_test.iloc[[0]])

if best_algorithm_name == "Baseline":
    sample_pred = int(X_test_unscaled["Credit_History"].iloc[0] >= 0.5)
elif best_algorithm_name == "Find-S":
    sample_pred = find_s_predict(df_fs_test.iloc[0], hypothesis, feature_cols_fs)
elif best_algorithm_name == "Decision Tree":
    sample_pred = dt_model.predict(X_test_unscaled.iloc[[0]])[0]
elif best_algorithm_name == "KNN":
    sample_pred = knn_model.predict(X_test_scaled.iloc[[0]])[0]
else:  # ANN
    sample_pred = int(ann_model.predict(X_test_scaled.iloc[[0]], verbose=0)[0][0] >= 0.5)

label_map = {1: "Approved", 0: "Rejected"}
print(f"Actual outcome    : {label_map[y_test.iloc[0]]}")
print(f"Predicted outcome : {label_map[sample_pred]}")

## Conclusion

- The `Loan_ID` column was removed as it carries no predictive information.
- Missing values in numeric columns were imputed with training-set medians; categorical columns used training-set modes.
- A single train-test split was selected using Decision Tree as a reference, then kept fixed for the five-algorithm comparison.
- Baseline, Find-S, Decision Tree, KNN, and ANN were implemented with separate model objects and explicit `.fit()` / `.predict()` calls.
- Preprocessing is performed manually; **the notebook does not use `Pipeline`**.
- The highest-F1 model is selected, while accuracy, precision, recall, and the confusion matrix provide additional evaluation context.

**Limitations**
- The dataset is small (~614 rows); results may vary across random splits.
- Class imbalance (~69 % Approved) means accuracy alone is a misleading metric; F1-score is more informative.
- Find-S degenerates on noisy data — it is a pedagogical, not a production, algorithm.
- No cross-validation is used; a single hold-out split can be sensitive to the random state.
- Features like Gender and Married should not drive lending decisions in a real system due to fairness concerns.